# Lesson 18 · Vision Transformer (ViT): cut an image into patches, read it with a Transformer, then compare with a CNN

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/18-vision-transformer-en.ipynb)

This notebook accompanies [Lesson 18 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vision-transformer/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/18-vision-transformer.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
ViT (Dosovitskiy et al., 2020) reads an image the way one reads a sentence: it cuts the image into square patches,
turns each patch into a token, then passes the whole sequence through the Transformer blocks of Lesson 17. We write
a ViT ourselves in numpy, run two ViTs trained on real MNIST, then ask: without convolutions, how does the model know
which patch sits where, and what is the price of not building in any knowledge about space?

**What you'll do:**
1. Cut a digit into a grid of patches, count the tokens, the parameters and the multiply-accumulate operations, and
   check that “flatten the patch, then multiply by $E$” is exactly a convolutional layer.
2. Write the ViT forward pass yourself (the [CLS] token, position embeddings, Transformer blocks without a mask) and
   run the demo's two trained ViTs on the 10,000 test images.
3. Shuffle the order of the patches, then shuffle the pixels within each patch, to see what the position embeddings
   do.
4. Read the position embeddings and the attention of the MNIST ViT (computed here) and of a real ViT‑Ti/16
   (precomputed measurements).
5. Train a small ViT yourself, then compare ViT with LeNet and an MLP as the number of training images grows from
   1,000 to 55,000.

**Background:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/)
(multi-head self-attention, LayerNorm, the Transformer block), [Lesson 11 · Convolutional networks (CNNs)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/cnn/)
and [Lesson 12 · Pooling & classic CNN architectures](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/kien-truc-cnn/)
(LeNet, the model we compare against). **Run time:** under a minute on a CPU, including the first MNIST download
(~11 MB).

Every number in the notebook is computed by the cells below (or read from the very files the demo uses), and matches
the numbers in the demo. Training ViT, LeNet and the MLP at 6 dataset sizes × 3 seeds takes more than an hour on a
many-core machine, so the notebook loads the weights of the two trained ViTs and the table of results. The real
ViT‑Ti/16 doesn't run here: its weights are not shipped with the notebook, only the measurements precomputed from it.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import io
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def sci(x, digits=1):
    """A very small error in the English format: 3.3·10⁻¹⁶ (numbers from 0.001 up print as plain decimals)."""
    if x == 0:
        return "0"
    if abs(x) >= 1e-3:
        return num(x, 3 if abs(x) < 1 else 2)
    m, e = f"{x:.{digits}e}".split("e")
    return f"{num(float(m), digits)}·10{str(int(e)).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527), sci(3.3e-16))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. The cell also loads three files that the demo uses (`fetch_json`, with an MD5 check):
- `vit-mnist-weights.json`: two ViTs trained on 55,000 images (seed 0), one with position embeddings and one
  without, their weights rounded to float16, along with the first 64 test images;
- `vit-mnist.json`: the results of ViT, LeNet and the MLP with 1,000–55,000 training images × 3 seeds, and the side
  experiments;
- `vit-pretrained.json`: measurements inside the real ViT‑Ti/16 on 8 public-domain photos.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of the long training runs (the same files as the demo on the web). build.py replaces COURSE_DATA_URL
# with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
shipped = fetch_json("vit-mnist-weights.json", "74ab5fb70eee71f88aee5b93b4d9ccd7")  # 2 trained ViTs + 64 images
runs = fetch_json("vit-mnist.json", "00448109c07dc2e8aee1caad93f12b68")  # ViT / LeNet / MLP × dataset size × seed
pre = fetch_json("vit-pretrained.json", "003ceb0d264e7b2f21195b864801b8c3")  # measurements of the real ViT-Ti/16

The weights are stored as float16, base64-encoded. We decode them into two copies: float64, to compare decimal by
decimal with the script that trained them, and float32 (twice as fast), to run on the whole test set.

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xte = (test_img / 255.0).astype(np.float32)  # (10,000, 28, 28), values in [0, 1]
yte = test_lab.astype(np.int64)


def unpack(d, dtype):
    return np.frombuffer(base64.b64decode(d["b64"]), "<f2").reshape(d["shape"]).astype(dtype)


cfg = shipped["cfg"]
P, D, L, HEADS, MLP = cfg["P"], cfg["D"], cfg["L"], cfg["heads"], cfg["mlp"]
vit64 = {name: {k: unpack(v, np.float64) for k, v in m.items()} for name, m in shipped["models"].items()}
vit32 = {name: {k: unpack(v, np.float32) for k, v in m.items()} for name, m in shipped["models"].items()}
digits = np.frombuffer(base64.b64decode(shipped["digits"]["x"]), np.uint8).reshape(-1, 28, 28)
y64 = np.array(shipped["digits"]["y"])
assert np.array_equal(digits, test_img[:64]) and np.array_equal(y64, yte[:64])  # exactly the first 64 test images

print(f"training: {num(len(train_img), 0)} images, test: {num(len(test_img), 0)} images, each {train_img.shape[1:]}")
print(f"trained ViT: {P}×{P} patches, D = {D}, {L} blocks × {HEADS} heads, MLP {MLP}; two versions: {list(vit64)}")
n_runs = sum(len(g["sweep"]) for g in runs["groups"]) + 2 * len(runs["groups"])
print(
    f"results file: {len(runs['groups'])} configurations × 3 seeds, {n_runs} training runs in total (η search included)"
)
su = runs["setup"]
print(
    f"training-set sizes: {' · '.join(num(n, 0) for n in su['sizes'])} images; each run {num(su['steps'], 0)} AdamW steps, batch {su['batch']}"
)
print(
    f"weight decay {num(su['wd'], 2)}; η warms up over the first {pct(su['warm'] / su['steps'], 0)} of the steps, then decays along a cosine"
)

16 of the 64 test images the demo uses, each with its true label above it:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for ax, img, lab in zip(axes.flat, digits, y64):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Cutting the image into a grid of patches

The Transformer of Lesson 17 reads a sequence of tokens. To feed it an image, ViT cuts the $H \times W$ image into
non-overlapping square $P \times P$ patches, then lines them up in reading order (left to right, top to bottom):

$$N = \frac{HW}{P^2}$$

For a 28×28 digit and $P = 4$, we get a 7×7 grid = 49 patches, each flattened into a row vector of 16 numbers. The
cell below writes the cutting as a loop so it is easy to read, then rewrites it in numpy: the numpy version only
changes the shape of the array; then we check that the two ways give the same result.

In [ ]:
def patchify_loop(x, P):
    """One image (28, 28) → (N, P²): walk the patch grid row by row, flatten each patch row by row."""
    G = x.shape[0] // P
    rows = []
    for gi in range(G):  # row of the patch grid
        for gj in range(G):  # column of the patch grid
            rows.append(x[gi * P : (gi + 1) * P, gj * P : (gj + 1) * P].reshape(-1))
    return np.array(rows)


def patchify(X, P):
    """Many images (B, 28, 28) → (B, N, P²), in exactly the order of patchify_loop."""
    B, H, _ = X.shape
    G = H // P
    return X.reshape(B, G, P, G, P).transpose(0, 1, 3, 2, 4).reshape(B, G * G, P * P)


x0 = Xte[0]
assert np.array_equal(patchify_loop(x0, 4), patchify(Xte[:1], 4)[0])
for size in (4, 7, 14):
    pts = patchify(Xte[:1], size)[0]
    print(
        f"{size:>2}×{size:<2} patches: a {28 // size}×{28 // size} grid = {len(pts):>2} patches, {len(pts) + 1:>2} tokens "
        f"with [CLS]; {size * size:>3} numbers per patch; {int((pts.max(1) == 0).sum()):>2} patches are all background"
    )

The figure below puts the image (left; the blue lines are the grid of 4×4 patches) next to the sequence of tokens
the Transformer actually receives (right): the first, blue square is the [CLS] token (section 5), followed by the 49
patches in reading order. The patches that are all background are white squares with a gray border. They are still
tokens like every other patch, and still take up a row and a column of the attention matrix.

In [ ]:
pts = patchify(Xte[:1], P)[0]
fig = plt.figure(figsize=(10, 2.9))
ax = fig.add_axes([0.0, 0.04, 0.24, 0.84])
ax.imshow(x0, cmap="gray_r", vmin=0, vmax=1)
for k in range(1, 28 // P):
    ax.axhline(k * P - 0.5, color=BLUE, lw=0.9)
    ax.axvline(k * P - 0.5, color=BLUE, lw=0.9)
ax.set_title(f"digit {yte[0]}, a 7×7 grid of 4×4 patches", fontsize=10)
ax.axis("off")
per_row, step = 25, P + 1  # the 50 tokens laid out in 2 rows × 25 squares, 1 pixel apart
mosaic = np.full((2 * step, per_row * step), np.nan)
for t in range(1, 50):
    r_, c_ = divmod(t, per_row)
    mosaic[r_ * step : r_ * step + P, c_ * step : c_ * step + P] = pts[t - 1].reshape(P, P)
cmap = plt.get_cmap("gray_r").copy()
cmap.set_bad("white")
ax = fig.add_axes([0.28, 0.04, 0.72, 0.84])
ax.imshow(mosaic, cmap=cmap, vmin=0, vmax=1)
for t in range(50):
    r_, c_ = divmod(t, per_row)
    ax.add_patch(
        Rectangle((c_ * step - 0.5, r_ * step - 0.5), P, P, fc=BLUE if t == 0 else "none", ec=GREY, lw=0.7, alpha=0.9)
    )
ax.text(1.5, 1.5, "CLS", color="white", fontsize=6, ha="center", va="center", fontweight="bold")
ax.set_title("the 50 tokens: [CLS], then the 49 patches in reading order", fontsize=10)
ax.axis("off")
plt.show()

Real color images are cut exactly the same way. ViT‑B/16 takes 224×224 images with $C = 3$ color channels. Each
16×16 patch is $P^2 C$ numbers, and the number of tokens grows with the square as the patches get smaller:

In [ ]:
def n_tokens(img, P):
    return (img // P) ** 2 + 1  # N patches + [CLS]


print(f"ViT-B/16 at 224×224: each patch 16·16·3 = {16 * 16 * 3} numbers; {n_tokens(224, 16)} tokens")
print(f"fine-tuning at 384×384 (as in the paper): {n_tokens(384, 16)} tokens")
ratio = (n_tokens(224, 8) / n_tokens(224, 16)) ** 2
print(f"8×8 patches instead of 16×16: {n_tokens(224, 8)} tokens, a T×T attention matrix {num(ratio, 1)} times larger")
print(f"one token per pixel: {num(n_tokens(224, 1), 0)} tokens")
assert (16 * 16 * 3, n_tokens(224, 16), n_tokens(384, 16), n_tokens(224, 8)) == (768, 197, 577, 785)

Halve the side of the patch and the number of tokens grows 4-fold, so the attention matrix (one row and one column
per token) grows nearly 16-fold. This is why ViT uses large patches: with one token per pixel, a 224×224 image
becomes 50,177 tokens.

## 4 · Each patch becomes a token, and that is really a convolution

Each patch $x_p^i$ (a row vector of $P^2 C$ numbers) is multiplied by the same learned matrix
$E \in \mathbb{R}^{(P^2 C) \times D}$ to become the token $x_p^i E$ of $D$ numbers (our MNIST ViT: $D = 48$; ViT‑B:
768). Column $d$ of $E$ holds $P^2 C$ weights; rearranged into a $P \times P \times C$ block, it is a **filter**. So
“flatten the patch, then multiply by $E$” is exactly a convolutional layer (Lesson 11) with $D$ filters of size
$P \times P$, stride $P$ and no padding: the windows don't overlap. The original ViT code writes it exactly that way
(`nn.Conv(..., kernel_size=patch, strides=patch, padding="VALID")`).

The cell below takes $E$ from the trained model, computes the convolution with a loop over each output position,
then compares it with the matrix multiplication.

In [ ]:
E = vit64["pos"]["We"]  # (16, 48): the 16 pixels of a patch → the 48 numbers of the token
x = digits[0] / 255.0  # float64, as in the demo's script
G = 28 // P
conv = np.zeros((G, G, D))
for i in range(G):  # window (i, j): the stride is P, so the windows don't overlap
    for j in range(G):
        window = x[i * P : (i + 1) * P, j * P : (j + 1) * P]
        for d in range(D):  # filter d = column d of E, rearranged into 4×4
            conv[i, j, d] = (window * E[:, d].reshape(P, P)).sum()
matmul = patchify(x[None], P)[0] @ E  # (49, 48)
err = np.abs(conv.reshape(G * G, D) - matmul).max()
print(
    f"convolution, {D} filters of {P}×{P}, stride {P}, no padding: output {conv.shape}, flattened {conv.reshape(-1, D).shape}"
)
print(f"flatten the patches, then multiply by E: {matmul.shape}; largest difference {sci(err)}")
assert err < 1e-12

The two ways give the same result because they do exactly the same multiply-accumulate operations; only the indexing
differs. A color image only adds a channel axis: the patch is flattened into $P^2 C$ numbers in exactly the
(channel, row, column) order in which the filter is flattened. The cell below checks this on a random 64×64 color
image with 8 filters of 16×16×3, then prints the difference that the demo's script measured on the real ViT‑Ti/16
(192 filters of 16×16×3, on 8 photos).

In [ ]:
def patch_rows_rgb(img, P):
    """A color image (C, H, W) → (N, C·P·P): each patch flattened in (channel, row, column) order, like the weights of
    a convolutional layer."""
    C, H, _ = img.shape
    G = H // P
    return img.reshape(C, G, P, G, P).transpose(1, 3, 0, 2, 4).reshape(G * G, C * P * P)


rng = np.random.default_rng(0)
img = rng.random((3, 64, 64))
filters = rng.normal(0, 0.02, (8, 3, 16, 16))  # 8 filters of 16×16×3
conv3 = np.array(
    [
        [(filters * img[:, i * 16 : (i + 1) * 16, j * 16 : (j + 1) * 16]).sum((1, 2, 3)) for j in range(4)]
        for i in range(4)
    ]
)
E3 = filters.reshape(8, -1).T  # (768, 8): column d is filter d, flattened
err3 = np.abs(conv3.reshape(16, 8) - patch_rows_rgb(img, 16) @ E3).max()
print(f"color image: {E3.shape[0]} numbers per patch; largest difference {sci(err3)}")
print(
    f"the real ViT-Ti/16 (measured beforehand by the demo's script): largest difference {sci(pre['model']['convErr'])}"
)
assert err3 < 1e-12

Differences of $10^{-16}$ to $10^{-14}$ are only rounding errors from adding in a different order. So the first
layer of ViT *is* a convolutional layer; it is just the only one: the stride equals the filter size, so each pixel
belongs to exactly one token, and after it there are no more convolutions.

## 5 · The [CLS] token, position embeddings and the Transformer blocks

The sequence that enters the first block is equation (1) of the paper, written in the row convention:

$$z_0 = \big[\,x_{\text{class}};\; x_p^{1}E;\; \dots;\; x_p^{N}E\,\big] + E_{\text{pos}}$$

- $x_{\text{class}}$ is the **[CLS]** token: a learned vector of $D$ numbers, placed at the start of the sequence,
  like BERT's [CLS] token. It belongs to no patch; through the blocks it gathers information from every patch, and
  only its output at the last layer goes into the classifier.
- $E_{\text{pos}}$ holds $N + 1$ learned **position embeddings**, one vector per position, added to each row. At
  first they are small random numbers, with no information at all about the 2‑D grid.

Then come $L$ Transformer blocks as in Lesson 17, pre‑LN style: $h \leftarrow h + \mathrm{MSA}(\mathrm{LN}(h))$,
then $h \leftarrow h + \mathrm{MLP}(\mathrm{LN}(h))$. They differ from GPT in one respect: **there is no mask**. Every
token can see every token, including the patches that come after it.

The cell below writes the forward pass exactly as the script that trained the models does (`reel/vit/vit.py`):
LayerNorm with $\epsilon = 10^{-6}$, GELU in its tanh approximation. The function can also return the attention
(section 9) and a cache for backpropagation (section 10).

In [ ]:
C_GELU = float(np.sqrt(2 / np.pi))


def layer_norm(h, g, b, eps=1e-6):
    hc = h - h.mean(-1, keepdims=True)
    inv = 1 / np.sqrt((hc * hc).mean(-1, keepdims=True) + eps)
    return hc * inv * g + b, (hc * inv, inv)


def gelu(u):  # the tanh approximation, as in GPT-2
    t = np.tanh(C_GELU * (u + 0.044715 * u * u * u))
    return 0.5 * u * (1 + t), t


def forward(p, X, P=P, heads=HEADS, maps=False, cache=False):
    """ViT on images X (B, 28, 28) → logits (B, 10). maps=True also returns the attention (B, heads, T, T) of every
    block."""
    xp = patchify(X, P)
    B, N, _ = xp.shape
    Dm = p["We"].shape[1]
    dh = Dm // heads
    h = xp @ p["We"] + p["be"]  # each patch becomes a token: x_p E
    h = np.concatenate([np.broadcast_to(p["cls"], (B, 1, Dm)), h], 1)  # the [CLS] token goes first
    if "pos" in p:
        h = h + p["pos"]  # + E_pos
    T, saved, att = N + 1, [], []
    for l in range(sum(k.endswith(".Wqkv") for k in p)):
        w = {k.split(".", 1)[1]: v for k, v in p.items() if k.startswith(f"{l}.")}
        a, c1 = layer_norm(h, w["g1"], w["b1"])
        q, k, v = (a @ w["Wqkv"] + w["bqkv"]).reshape(B, T, 3, heads, dh).transpose(2, 0, 3, 1, 4)
        A = q @ k.transpose(0, 1, 3, 2) * (1 / dh**0.5)  # no mask: every token sees every token
        A = np.exp(A - A.max(-1, keepdims=True))
        A /= A.sum(-1, keepdims=True)
        o = (A @ v).transpose(0, 2, 1, 3).reshape(B, T, Dm)
        h = h + o @ w["Wo"] + w["bo"]  # h ← h + MSA(LN h)
        m, c2 = layer_norm(h, w["g2"], w["b2"])
        u = m @ w["W1"] + w["c1"]
        gu, t = gelu(u)
        h = h + gu @ w["W2"] + w["c2"]  # h ← h + MLP(LN h)
        if maps:
            att.append(A)
        if cache:
            saved.append((a, c1, q, k, v, A, o, m, c2, u, gu, t))
    hf, cf = layer_norm(h[:, 0], p["gf"], p["bf"])  # read only the [CLS] token at the last layer
    z = hf @ p["Wh"] + p["bh"]
    if cache:
        return z, (xp, saved, hf, cf)
    return (z, att) if maps else z


def predict(p, X, bs=500, **kw):
    return np.concatenate([forward(p, X[i : i + bs], **kw).argmax(1) for i in range(0, len(X), bs)])

## 6 · Counting parameters and multiply-accumulate operations

Before running anything, we count. A Transformer block has four $D \times D$ matrices (Q, K, V and the output
projection; here Q, K, V are merged into one $D \times 3D$ matrix), an MLP $D \to M \to D$, two LayerNorms (each with
a scale vector and a shift vector), and the biases: $4D^2 + 2DM + 9D + M$ parameters. Outside the blocks there are
also $E$, [CLS], the position embeddings, the final LayerNorm and the classifier.

The multiply-accumulate (MAC) operations for one image fall into two parts. The projections (making the tokens,
Q/K/V, the output, the MLP) grow with the number of tokens $T$. The attention part (the scores $QK^\top$ and the
weighted sum $AV$, each $T^2 D$ MACs per block) grows with $T^2$. As in earlier lessons, we don't count the biases,
LayerNorm, softmax or GELU.

In [ ]:
def block_params(Dm, M):
    return 4 * Dm * Dm + 2 * Dm * M + 9 * Dm + M


def vit_params(P, Dm, L, M, img=28, C=1, classes=10, pos=True):
    N = (img // P) ** 2
    parts = {
        "E (patch projection)": P * P * C * Dm + Dm,
        "[CLS]": Dm,
        "position embeddings": (N + 1) * Dm if pos else 0,
        f"{L} Transformer blocks": L * block_params(Dm, M),
        "final LN + classifier": 2 * Dm + Dm * classes + classes,
    }
    return parts, sum(parts.values())


def vit_macs(P, Dm, L, M, img=28, C=1, classes=10):
    N = (img // P) ** 2
    T = N + 1
    attention = L * 2 * T * T * Dm
    rest = N * P * P * C * Dm + L * (4 * T * Dm * Dm + 2 * T * Dm * M) + Dm * classes
    return attention, rest


parts, n_pos = vit_params(P, D, L, MLP)
for k, v in parts.items():
    print(f"{k:<22} {num(v, 0):>7}")
n_shipped = {name: sum(a.size for a in p.values()) for name, p in vit64.items()}
_, n_nopos = vit_params(P, D, L, MLP, pos=False)
print(
    f"total: {num(n_pos, 0)} parameters (counted in the file: {num(n_shipped['pos'], 0)}); without positions: {num(n_nopos, 0)}"
)
assert n_pos == n_shipped["pos"] == 79690 and n_nopos == n_shipped["nopos"] == 77290
print(f"share of the Transformer blocks: {pct(parts[f'{L} Transformer blocks'] / n_pos, 1)}")

95.2% of the parameters are in the 4 Transformer blocks; $E$ has only 816. Now the compute, compared with two models
from earlier lessons: Lesson 12's LeNet and the 784 → 128 → 64 → 10 MLP of Lessons 01–10.

In [ ]:
lenet_params = (5 * 5 * 1 * 6 + 6) + (5 * 5 * 6 * 16 + 16) + (400 * 120 + 120) + (120 * 84 + 84) + (84 * 10 + 10)
lenet_macs = 28 * 28 * 25 * 6 + 10 * 10 * 150 * 16 + 400 * 120 + 120 * 84 + 84 * 10
mlp_params = (784 * 128 + 128) + (128 * 64 + 64) + (64 * 10 + 10)
mlp_macs = 784 * 128 + 128 * 64 + 64 * 10
macs = {}
for size in (4, 7, 14):
    att, rest = vit_macs(size, D, L, MLP)
    macs[size] = att + rest
    print(
        f"ViT, {size:>2}×{size:<2} patches ({(28 // size) ** 2 + 1:>2} tokens): {num(vit_params(size, D, L, MLP)[1], 0):>6} parameters, "
        f"{num(att + rest, 0):>9} MACs, attention takes {pct(att / (att + rest), 1)}"
    )
print(f"LeNet (Lesson 12):              {num(lenet_params, 0):>6} parameters, {num(lenet_macs, 0):>9} MACs")
print(f"MLP 784 → 128 → 64 → 10:       {num(mlp_params, 0):>6} parameters, {num(mlp_macs, 0):>9} MACs")
assert (macs[4], macs[7], macs[14], lenet_macs, mlp_macs) == (4684512, 1402464, 416352, 416520, 109184)
assert (lenet_params, mlp_params) == (61706, 109386)
p4, p7 = (list(vit_params(size, D, L, MLP)[0].values()) for size in (4, 7))
assert (p7[0], p7[2]) == (p4[2], p4[0]) == (2400, 816) and sum(p7) == sum(p4)  # E and the position table swap sizes

7×7 patches give exactly the same number of parameters as 4×4, because the two sizes swap roles: a 4×4 patch has 16
pixels and there are 49 patches; a 7×7 patch has 49 pixels and there are 16 patches. $E$ grows from 816 to 2,400
parameters, and the table of position embeddings shrinks by just as much, from 2,400 to 816. With 17 tokens instead
of 50, 7×7 patches cost less than a third. 14×14 patches give only 4 patches and cost about as much as
LeNet. The ViT with 4×4 patches costs more than 11 times as much as LeNet for one 28×28 digit: the projections are
applied to every token, so the more tokens, the more expensive.

The same two functions can count a real ViT. ViT‑Ti/16 (Touvron et al., 2020: $D = 192$, 12 blocks × 3 heads,
$M = 768$) on 224×224 color images with 1,000 classes, compared with the number of parameters in the real weights
file, which the demo's script counted:

In [ ]:
parts_ti, n_ti = vit_params(16, 192, 12, 768, img=224, C=3, classes=1000)
att_ti, rest_ti = vit_macs(16, 192, 12, 768, img=224, C=3, classes=1000)
print(f"ViT-Ti/16: {num(n_ti, 0)} parameters by the formula, {num(pre['model']['params'], 0)} in the real weights file")
print(
    f"{num((att_ti + rest_ti) / 1e9, 2)} billion MACs per image, attention takes {pct(att_ti / (att_ti + rest_ti), 1)}"
)
assert n_ti == pre["model"]["params"] == 5717416

The formula matches to the last parameter. At 197 tokens, attention takes only about a seventh of the compute; most
of it is still matrix multiplication on each token.

## 7 · Running the two trained ViTs

The two models below are identical in architecture and training (55,000 images, AdamW, 8,000 steps, seed 0), and
differ in one thing only: `pos` adds learned position embeddings, `nopos` doesn't. First we check that our `forward`
computes exactly what the script that trained them computes: we compare the 10 logits of the first test image (a 7)
with the values the script printed when it exported the models, then count how many of the demo's 64 images are
correct.

In [ ]:
REF = {  # logits of test image #0, computed by reel/vit/mnist.py in float64 when exporting the models (check.json)
    "pos": [-1.9142892619, -0.7331674863, -0.4271250327, 1.7714222114, -3.7937051965,
            -0.0626626205, -2.9580504745, 9.6289179084, -2.0023079834, 0.3459495349],
    "nopos": [-6.4796454084, 1.7445987811, 1.4421554203, -1.6455837039, 1.0693166588,
              -1.3073301918, -2.3708101102, 8.3913805659, -5.1736052158, 2.5766033003],
}  # fmt: skip
X64 = digits / 255.0
z64 = {name: forward(p, X64) for name, p in vit64.items()}
for name in vit64:
    gap = np.abs(z64[name][0] - REF[name]).max()
    ok = int((z64[name].argmax(1) == y64).sum())
    print(f"{name:>5}: logits of image #0 = [{', '.join(num(v, 2) for v in z64[name][0])}]")
    print(f"       largest deviation from the script {sci(gap)}; {ok}/64 of the demo's images correct")
    assert gap < 1e-8
assert [int((z64[n].argmax(1) == y64).sum()) for n in ("pos", "nopos")] == [64, 61]

The logits match the script to the 10th decimal place. Now we run all 10,000 test images with the float32 copy
(faster: a few seconds per model), then compare with the accuracy the script measured for seed 0 and the range over
the 3 seeds.

In [ ]:
def group(gid, n=55000):
    """One configuration in the results file (model id, number of training images); g["seeds"] holds its 3 seeds."""
    return next(g for g in runs["groups"] if g["id"] == gid and g["n"] == n)


def span(v, digits=2):
    return f"{pct(min(v), digits)}–{pct(max(v), digits)}"


pred = {name: predict(p, Xte) for name, p in vit32.items()}
acc = {name: (pr == yte).mean() for name, pr in pred.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    seeds = [s["test"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: {pct(acc[name])} correct on the 10,000 test images; script: {pct(seeds[0])} (seed 0), {span(seeds)} (3 seeds)"
    )
    assert abs(acc[name] - seeds[0]) < 3e-4, "differs from the demo: check the data again"
print(f"without the position embeddings: {num(100 * (acc['pos'] - acc['nopos']), 1)} percentage points lost")

The two models differ only by 2,400 position parameters, yet one gets 97.83% right and the other 89.07%. The next
section looks into why.

## 8 · Shuffling the patches: what do the position embeddings do?

Self-attention compares each pair of tokens by their content ($q_i \cdot k_j$) and adds up the $v_j$. Nothing in
that computation says where token $j$ sits in the image. The only position information is $E_{\text{pos}}$, added at
the start.

### 🤔 Predict first
Keep the content of each patch, but shuffle the order of the 49 patches (a different shuffle for each image). Of the
demo's 64 images, how many will the model without positions get right (61 before shuffling)? And the model with
positions (64 before shuffling)?

In [ ]:
def unpatchify(xp, P=P):
    """The inverse of patchify: (B, N, P²) → (B, 28, 28)."""
    B, G = len(xp), 28 // P
    return xp.reshape(B, G, G, P, P).transpose(0, 1, 3, 2, 4).reshape(B, 28, 28)


def shuffle_patches(X, orders, P=P):
    """Square k of the grid receives patch orders[b, k]: each image b has its own permutation."""
    return unpatchify(patchify(X, P)[np.arange(len(X))[:, None], orders], P)


rng = np.random.default_rng(11)  # exactly the permutations the script used to check the demo
Xs64 = shuffle_patches(X64, np.array([rng.permutation(49) for _ in range(64)]))
REF_SHUF = [-2.0778332685, 3.0591328714, -0.183120002, 0.6325104861, 1.1377082091,
            0.3617171861, -2.8125161098, 2.7310975398, -1.2453949321, -1.0833118026]  # fmt: skip
zs64 = {name: forward(p, Xs64) for name, p in vit64.items()}
for name in vit64:
    ok = int((zs64[name].argmax(1) == y64).sum())
    same = int((zs64[name].argmax(1) == z64[name].argmax(1)).sum())
    diff = np.abs(zs64[name] - z64[name]).max()
    print(
        f"{name:>5}: {ok}/64 correct after shuffling; {same}/64 predictions unchanged; largest logit change {sci(diff)}"
    )
assert np.abs(zs64["pos"][0] - REF_SHUF).max() < 1e-8  # image #0 after shuffling, as the script computed it
assert np.abs(zs64["nopos"] - z64["nopos"]).max() < 1e-10
assert [int((zs64[n].argmax(1) == y64).sum()) for n in ("pos", "nopos")] == [19, 61]

The model without positions gives **exactly the same logits**; the difference of $10^{-15}$ comes only from adding
in a different order. Without $E_{\text{pos}}$, each token depends only on the content of its patch. LayerNorm,
Q/K/V and the MLP are applied to each token separately, and attention is a weighted sum over the *set* of tokens. So
swapping the input tokens swaps the output tokens in exactly the same way (*permutation equivariance*). The [CLS]
token is not a patch, so it stays put, and its output gathers from every patch, so it doesn't change. This model sees
only a **bag of patches**.

The model with positions collapses: from 64/64 down to 19/64. It has learned “the patch at position $k$ has this
content”, so when the patches move, it reads a different digit. On the first 500 test images, with exactly the
permutations the script used when it measured the whole test set (`default_rng(7)`, one permutation per image):

In [ ]:
N_SUB = 500  # the first test images; the script measures all 10,000
rng = np.random.default_rng(7)
X_patch = shuffle_patches(Xte[:N_SUB], np.array([rng.permutation(49) for _ in range(N_SUB)]))
pred_patch = {name: predict(p, X_patch) for name, p in vit32.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    a_sub = (pred_patch[name] == yte[:N_SUB]).mean()
    stored = [s["patches"]["acc"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: patches shuffled, {pct(a_sub, 1)} correct on {num(N_SUB, 0)} images; all 10,000 images (script, 3 seeds): {span(stored)}"
    )
same_nopos = int((pred_patch["nopos"] == pred["nopos"][:N_SUB]).sum())
print(f"nopos: {same_nopos}/{N_SUB} predictions the same as on the original images")
assert same_nopos >= N_SUB - 1  # a bag of patches (float32: a near-exact tie can flip from rounding)

The model with positions keeps only 18.76–21.01% on the whole test set (3 seeds), just twice the level of random
guessing (1 in 10). So the MNIST model *uses* position a great deal. Section 9 will show that it uses position in an
unexpected way.

### 🤔 Predict first
Now the other way around: keep each patch in its place, but shuffle the 16 pixels *within* each patch (one
permutation per image, shared by every patch of that image). Which model copes better: the one with positions or the
one without?

In [ ]:
def shuffle_pixels(X, perms, P=P):
    """Shuffle the P² pixels within every patch of image b with the same permutation perms[b]."""
    return unpatchify(np.take_along_axis(patchify(X, P), perms[:, None, :], axis=2), P)


rng = np.random.default_rng(7)  # as in the script
X_pix = shuffle_pixels(Xte[:N_SUB], np.array([rng.permutation(P * P) for _ in range(N_SUB)]))
pred_pix = {name: predict(p, X_pix) for name, p in vit32.items()}
for name, gid in (("pos", "vit4"), ("nopos", "vit4-nopos")):
    a_sub = (pred_pix[name] == yte[:N_SUB]).mean()
    stored = [s["pixels"]["acc"] for s in group(gid)["seeds"]]
    print(
        f"{name:>5}: pixels shuffled, {pct(a_sub, 1)} correct on {num(N_SUB, 0)} images; all 10,000 images (script): {span(stored)}"
    )
ink_kept = np.allclose(patchify(X_pix, P).sum(2), patchify(Xte[:N_SUB], P).sum(2), atol=1e-5)
print("the amount of ink in each patch is unchanged after shuffling the pixels:", ink_kept)
assert ink_kept

The model with positions copes much better: 65.06–66.60% versus 11.79–13.67% (whole test set, 3 seeds). The model
without positions has only the shape of the strokes within each patch to rely on, so shuffling the pixels within the
patches takes away almost everything. The model with positions has another source of information: how much ink the
patch at each position has, and shuffling the pixels keeps that amount of ink (the cell above prints `True`).

Let's remove the stroke shapes entirely: replace each patch with a uniform gray, as dark as that patch's average. The
image is then just a coarse 7×7 grid of pixels, and the model without positions sees only a bag of 49 gray squares.

In [ ]:
coarse = unpatchify(np.repeat(patchify(Xte[:N_SUB], P).mean(2, keepdims=True), P * P, axis=2))
for name, p in vit32.items():
    print(
        f"{name:>5}: only the 7×7 grid of intensities left, {pct((predict(p, coarse) == yte[:N_SUB]).mean(), 1)} correct"
    )

fig, axes = plt.subplots(1, 4, figsize=(8, 2.6))
k = 1  # test image #1, a 2
for ax, (im, title) in zip(
    axes,
    [
        (Xte[k], "original\n"),
        (X_patch[k], "patch order\nshuffled"),
        (X_pix[k], "pixels shuffled\nwithin patches"),
        (coarse[k], "only each patch's\nintensity kept"),
    ],
):
    ax.imshow(im, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

The figure above shows four versions of a 2: the original, the patches shuffled, the pixels shuffled within the
patches, and only the mean intensity of each patch kept. To the human eye, the second image is a jumble of loose
strokes, while the last one still faintly looks like a 2. The model with positions reads it the same way: with only
the 7×7 grid of intensities, it still gets about seven tenths of the images right. Most of what it knows lies in
*where the ink is*, not in the shape of the strokes within each patch.

## 9 · What do the position embeddings learn?

The ViT paper plots the cosine similarity (cos) between the position embeddings of every pair of patches
(Figure 7): after training, nearby patches have more similar vectors, and so do patches in the same row or the same
column. The model learns the 2‑D grid by itself, even though it is only given 196 vectors in a sequence.

### 🤔 Predict first
The MNIST model with positions collapses when the patches are shuffled, so it relies heavily on position. Do its
position embeddings, then, learn the 2‑D grid: is the cos between two adjacent patches clearly larger than the cos
between two distant patches?

In [ ]:
def grid_stats(pos, G):
    """cos between the position embeddings of every pair of patches (a G×G grid), and the measures the demo's script
    uses."""
    pn = pos / np.linalg.norm(pos, axis=1, keepdims=True)
    cos = pn @ pn.T
    r, c = np.divmod(np.arange(G * G), G)
    dist = np.hypot(r[:, None] - r[None], c[:, None] - c[None])  # distance between two patches, in patches
    off = ~np.eye(G * G, dtype=bool)
    best = np.where(off, cos, -9).argmax(1)  # the patch with the most similar position embedding (other than itself)
    return cos, {
        "near": cos[dist == 1].mean(),  # two patches sharing an edge
        "far": cos[dist >= 4].mean(),  # two patches ≥ 4 patches apart
        "corr": np.corrcoef(cos[off], -dist[off])[0, 1],  # correlation between cos and closeness
        "nearest": int((np.maximum(abs(r - r[best]), abs(c - c[best])) <= 1).sum()),  # the most similar is adjacent
    }


cos_m, st = grid_stats(vit64["pos"]["pos"][1:], 7)  # drop the position of [CLS]
A_pos = runs["analysis"]["pos"]
print(
    f"MNIST ViT: cos of adjacent patches {num(st['near'], 3)}, ≥ 4 patches apart {num(st['far'], 3)}; "
    f"cos–closeness correlation {num(st['corr'], 3)}; most similar patch is adjacent: {st['nearest']}/49"
)
assert abs(st["near"] - A_pos["near"]) < 1e-3 and abs(st["corr"] - A_pos["corr"]) < 1e-3
assert st["nearest"] == A_pos["nearestNeighbour"] == 20
ever_ink = patchify(train_img[:10000] / 255.0, P).mean((0, 2)) >= 1e-3  # drop the nearly always empty border patches
cos_ink = cos_m[np.ix_(ever_ink, ever_ink)]
ri, ci = np.divmod(np.flatnonzero(ever_ink), 7)
d_ink = np.hypot(ri[:, None] - ri[None], ci[:, None] - ci[None])
print(
    f"only the {int(ever_ink.sum())} positions that ever have ink: cos of adjacent patches {num(cos_ink[d_ink == 1].mean(), 3)}, "
    f"cos–closeness correlation {num(np.corrcoef(cos_ink[d_ink > 0], -d_ink[d_ink > 0])[0, 1], 3)}"
)

No. The cos between two adjacent patches is only 0.138, almost the same as the cos between two distant patches, and
the correlation between the cos and how close two patches are is nearly 0. The result doesn't change when we drop
the 7 border patches that are almost always empty (the second line: only the 42 positions that ever have ink). The
MNIST model uses position a great deal (shuffling the patches makes it collapse) without learning the grid: each
position embedding is like a separate *house number*, not a coordinate. The model learns “what ink in square 24
means”, the way an MLP learns each pixel, without knowing that square 24 is next to square 25. MNIST digits are
always centered, so learning each position separately like this is enough.

The real ViT‑Ti/16 (pretrained on ImageNet‑21k, 14 million images, then fine-tuned on ImageNet‑1k) is different. The
demo's script measured the cos between its 196 position embeddings; the file `vit-pretrained.json` stores the whole
196 × 196 matrix (each number rounded to one byte) and the measures:

In [ ]:
cos_t = np.frombuffer(base64.b64decode(pre["pos"]["cos"]), np.uint8).reshape(196, 196) / 255 * 2 - 1
pt = pre["pos"]
print(
    f"ViT-Ti/16: cos of adjacent patches {num(pt['near'], 3)}; same row or same column, ≥ 4 patches apart {num(pt['lineFar'], 3)}; "
    f"different row and column, ≥ 4 patches apart {num(pt['offFar'], 3)}"
)
print(f"cos–closeness correlation {num(pt['corr'], 3)}; most similar patch is adjacent: {pt['nearestNeighbour']}/196")

fig, axes = plt.subplots(1, 4, figsize=(10, 2.9))
for ax, (cos, G_, sel, name) in zip(
    axes,
    [
        (cos_m, 7, 24, "MNIST ViT"),
        (cos_m, 7, 12, "MNIST ViT"),
        (cos_t, 14, 65, "ViT‑Ti/16"),
        (cos_t, 14, 142, "ViT‑Ti/16"),
    ],
):
    im = ax.imshow(cos[sel].reshape(G_, G_), cmap=DIVERGING, vmin=-1, vmax=1)
    r_, c_ = divmod(sel, G_)
    ax.add_patch(Rectangle((c_ - 0.5, r_ - 0.5), 1, 1, fc="none", ec="black", lw=1.5))
    ax.set_title(f"{name}: row {r_ + 1}, column {c_ + 1}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.02, label="cos with the selected patch")
fmt_axes(fig)
plt.show()

Each square is the cos between the position embedding of the patch framed in black and the position embedding of the
patch in that square (blue: similar, orange: opposite). The two plots on the left (MNIST, a 7×7 grid) are almost
noise: the squares next to the selected one are no bluer than the others. The two plots on the right (ViT‑Ti, a 14×14
grid) show a blue spot around the selected patch, stretching along its row and its column. No one told ViT‑Ti that
the image is a 14×14 grid; it learned that by itself from the data, because in real photos an object can be anywhere
and two adjacent patches often belong to the same object.

The ViT paper also tried building 2‑D positions in (Table 8, ViT‑B/16 pretrained, then evaluated on ImageNet with a
linear classifier trained on 5 images per class): no positions 61.4%, learned 1‑D positions 64.2%, 2‑D positions
64.0%, relative positions 64.0%. Having positions matters; which kind hardly does.

### Where does the MNIST ViT's attention look?

The figure below shows the attention of the [CLS] token to the 49 patches in each block (averaged over the 3 heads)
for the first test image, a 7. The darker the blue, the more information [CLS] takes from that patch.

In [ ]:
_, att0 = forward(vit64["pos"], X64[:1], maps=True)
BLUES = LinearSegmentedColormap.from_list("white_blue", ["#ffffff", BLUE])
fig, axes = plt.subplots(1, L, figsize=(9, 2.6))
for l, ax in enumerate(axes):
    m = att0[l][0, :, 0, 1:].mean(0).reshape(7, 7)  # [CLS] → 49 patches, averaged over the 3 heads
    ax.imshow(m, cmap=BLUES, vmin=0, extent=(-0.5, 27.5, 27.5, -0.5))
    ax.contour(X64[0], levels=[0.5], colors=ORANGE, linewidths=1.2)  # the digit's strokes (orange)
    ax.set_title(f"block {l + 1}: {pct(m.max(), 0)} on 1 patch", fontsize=9)
    ax.axis("off")
plt.show()
inked = patchify(X64[:1], P)[0].max(1) > 0  # patches with ink
cls_maps = [att0[l][0, :, 0, 1:].mean(0) for l in range(L)]  # [CLS] → 49 patches, averaged over the 3 heads
print(
    f"most attention on one patch, per block: {', '.join(pct(m.max(), 0) for m in cls_maps)} (even split: {pct(1 / 49, 0)})"
)
print(
    f"the {int(inked.sum())}/49 patches with ink ({pct(inked.mean(), 0)}) receive: "
    + ", ".join(pct(m[inked].sum() / m.sum(), 0) for m in cls_maps)
)
print("[CLS] looking at itself, per block:", ", ".join(pct(att0[l][0, :, 0, 0].mean()) for l in range(L)))

The orange line is the outline of the strokes. In each block, [CLS] concentrates its attention on a few patches: the
patch that receives the most gets 14–34%, whereas an even split would give each patch only 2%. In blocks 1 and 4, the
darkest patches lie along the stem of the 7. But the 16 patches with ink (33% of the patches) receive only 39–53% of
the attention: [CLS] also reads a lot from the background patches, and hardly looks at itself. An attention map only
tells where information flows to [CLS] from in one block, not the reason for the prediction: the patches have
already mixed each other's information in the earlier blocks.

To measure whether a head looks near or far, the ViT paper uses the **mean attention distance**: for each patch, the
average distance (in px, between the centers of the two patches) to the patches it looks at, weighted by attention.
It is like the receptive-field size of a CNN (Lesson 12). The cell below measures it on 1,000 test images, as the
demo's script does. If a head looks evenly over the whole image, the number is the average distance between two
arbitrary patches.

In [ ]:
G = 28 // P
r, c = np.divmod(np.arange(G * G), G)
dist_px = np.hypot(r[:, None] - r[None], c[:, None] - c[None]) * P  # px between the centers of two patches


def attention_distance(p, X, bs=250):
    """(L, heads): the mean attention distance, patch → patch only (drop [CLS], renormalize so each row sums to 1)."""
    out = np.zeros((L, HEADS))
    for i in range(0, len(X), bs):
        for l, A in enumerate(forward(p, X[i : i + bs], maps=True)[1]):
            pp = A[:, :, 1:, 1:].astype(np.float64)
            pp /= pp.sum(-1, keepdims=True)
            out[l] += (pp * dist_px).sum(-1).mean(-1).sum(0)
    return out / len(X)


N_ATT = runs["setup"]["nAtt"]  # the first 1,000 test images, as in the script
dist_mnist = {name: attention_distance(p, Xte[:N_ATT]) for name, p in vit32.items()}
uniform = dist_px.mean()
print(f"looking evenly over the whole image: {num(uniform, 1)} px")
for name, d in dist_mnist.items():
    print(f"{name:>5}: " + " · ".join(f"block {l + 1}: " + ", ".join(num(v, 1) for v in d[l]) for l in range(L)))
    assert np.abs(d - np.array(runs["analysis"][name]["dist"])).max() < 2e-3, "differs from the demo"
assert abs(uniform - runs["analysis"]["pos"]["uniform"]) < 1e-3

Every head, in every block, of both models looks 12–15 px away, close to the level of looking evenly over the whole
image (14.4 px). No head specializes in looking around its own patch like a convolution filter. With 55,000 small,
centered digits, the model doesn't need such heads to reach 97.83%.

### The real ViT‑Ti/16: near-looking heads, far-looking heads

The ViT paper (Figure 7, right) finds something different in large models trained on a lot of data: already in the
first block, some heads look over the whole image, while others look only around their own patch, like a
convolution; the deeper the block, the farther every head looks. Raghu et al. (2021) found that when trained only on
ImageNet, the early layers of ViT‑L/16 and ViT‑H/14 do not learn those near-looking heads. The demo's script measured
exactly this number on ViT‑Ti/16 with 8 photos (224×224, 16 px patches). The figure below plots it from the file
`vit-pretrained.json`: one marker per head (circle, square, triangle); the dashed gray line is the level of looking
evenly over the whole image.

In [ ]:
dist_ti, dist_ti_clean = np.array(pre["dist"]), np.array(pre["distClean"])  # (12 blocks, 3 heads), mean of 8 photos
MARKS = [("o", BLUE), ("s", ORANGE), ("^", GREEN)]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, d, title in (
    (axes[0], dist_ti, "all patches"),
    (axes[1], dist_ti_clean, "high-norm patches removed (see below)"),
):
    for h, (mk, col) in enumerate(MARKS):
        ax.plot(np.arange(1, 13), d[:, h], marker=mk, color=col, lw=1.2, ms=6, label=f"head {h + 1}")
    ax.axhline(pre["uniform"], color=GREY, ls="--", lw=1)
    ax.set(xlabel="block", title=f"ViT‑Ti/16, {title}", xticks=range(1, 13), ylim=(0, 160))
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("attention distance (px)")
axes[0].legend(frameon=False, fontsize=9, loc="lower right")
fmt_axes(fig)
plt.show()
print(f"looking evenly over the 224×224 image: {num(pre['uniform'], 1)} px")
print("block 1, three heads:", ", ".join(f"{num(v, 0)} px" for v in dist_ti[0]))
print("block 12, three heads:", ", ".join(f"{num(v, 0)} px" for v in dist_ti[11]))
print("block 12, without the high-norm patches:", ", ".join(f"{num(v, 0)} px" for v in dist_ti_clean[11]))

In block 1, head 3 looks only within a radius of about 26 px (less than two patches), while head 2 looks 81 px away.
The deeper the block, the farther all three heads look. From block 10 on (left plot), all three heads even look
farther than the level of looking evenly over the image: the attention piles up on a few patches very far away.
Remove those patches (right plot) and the distance in block 12 comes back to about 115 px. Which patches are they?

### The four corners of the image attract attention

The script measured the norm (length) of each patch token after every block, and flagged the tokens whose norm is
more than 5 times the median. The cell below reads the results for the 8 photos: the class the model predicts, the
high-norm patches in block 12 (patch 0 is the top-left corner, 13 the top-right, 182 the bottom-left, 195 the
bottom-right), and the share of attention [CLS] gives them in block 12.

In [ ]:
PHOTO = {  # English names of the 8 photos, by Commons file title (the file's labels are the Vietnamese demo's)
    "Golden Retriever Dukedestiny01 drvd.jpg": "golden retriever",
    "Brown tabby cat 2018 G1.jpg": "tabby cat",
    "Crusaders School Bus, Chalmette, Louisiana.jpg": "school bus",
    "Espresso BW 1.jpg": "cup of espresso",
    "Red eyed tree frog edit2.jpg": "red-eyed tree frog",
    "Kite sailboat dinghy.jpg": "sailboat",
    "Pizza pan and pizza.jpg": "pizza in a pan",
    "Sunflower head 2015 G1.jpg": "sunflower",
}
CORNERS = {0, 13, 182, 195}
first_high, mass, ratios = [], [], []
for ph in pre["photos"]:
    norms = np.frombuffer(base64.b64decode(ph["norms"]), np.uint8).reshape(12, 196) / 255 * ph["normsMax"]
    ratios.append(norms[11].max() / ph["normMedian"][11])
    first_high.append(next(l for l, hs in enumerate(ph["high"]) if hs) + 1)
    mass.append(ph["highMassCls"][11])
    k_top, p_top = ph["top"][0]
    verdict = "correct" if k_top in ph["expect"] else ("no correct class" if not ph["expect"] else "wrong")
    print(
        f"{PHOTO[ph['title']]:<22} predicts “{pre['classes'][str(k_top)]}” {pct(p_top, 1):>6} ({verdict}); "
        f"high-norm patches in block 12: {ph['high'][11]}, [CLS] gives them {pct(ph['highMassCls'][11], 1)}"
    )
print(f"the first high-norm patches appear in block {sorted(set(first_high))}; from then to block 12 they are always corners:",
      all(set(h) <= CORNERS for ph in pre["photos"] for h in ph["high"]))  # fmt: skip
print(f"in block 12, the largest norm is {num(min(ratios), 0)}–{num(max(ratios), 0)} times the median")
print(
    f"[CLS] in block 12 gives {pct(min(mass), 0)}–{pct(max(mass), 0)} of its attention to those 3–4 corner patches (4 patches = {pct(4 / 196, 0)} of the patches)"
)

In all 8 photos, whether they show a dog, a bus or a coffee cup, from block 8 on there are 3–4 tokens exactly at the
corners of the image with norms tens of times larger than the other tokens. In block 12, [CLS] gives those 3–4
patches between 22% and 53% of its attention, even though they are only 2% of the patches and carry almost no
information about the object in the image. The figure below shows this on two photos: the original, the attention
of [CLS] in block 4 and in block 12 (averaged over the 3 heads; the dashed orange frames are the high-norm patches),
and *attention rollout* (Abnar & Zuidema, 2020: the attention multiplied through the 12 blocks, with the skip
connections added in).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5.3))
for row, i in enumerate([0, 2]):  # golden retriever, school bus
    ph = pre["photos"][i]
    cls = np.frombuffer(base64.b64decode(ph["cls"]), np.uint8).reshape(12, 3, 196) / 255
    cls = cls * np.array(ph["clsMax"])[:, :, None]  # each map is stored relative to its own maximum
    roll = np.frombuffer(base64.b64decode(ph["rollout"]), np.uint8) / 255 * ph["rolloutMax"]
    axes[row, 0].imshow(plt.imread(io.BytesIO(base64.b64decode(ph["jpeg"])), format="jpeg"))
    axes[row, 0].set_title(PHOTO[ph["title"]], fontsize=9)
    maps = [
        (cls[3].mean(0), 3, "[CLS] in block 4"),
        (cls[11].mean(0), 11, "[CLS] in block 12"),
        (roll, 11, "rollout over 12 blocks"),
    ]
    for ax, (m, blk, title) in zip(axes[row, 1:], maps):
        ax.imshow(m.reshape(14, 14), cmap=BLUES, vmin=0, extent=(0, 224, 224, 0))
        for j in ph["high"][blk]:
            r_, c_ = divmod(j, 14)
            ax.add_patch(Rectangle((c_ * 16, r_ * 16), 16, 16, fc="none", ec=ORANGE, lw=1.8, ls="--"))
        ax.set_title(title, fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

In block 4, the attention of [CLS] is still spread over the whole image, and there are no high-norm patches yet. In
block 12, almost all of it piles up in the corners (orange frames). Rollout mixes every block: for the bus it lights
up on the body of the bus, for the dog it is still spread almost evenly. It is the path the information takes, not
an explanation of the prediction.

Darcet et al. (2023) found the same phenomenon in large ViTs (DeiT‑III and OpenCLIP from ViT‑B size up, DINOv2 from
ViT‑L size up): about 2% of the tokens, usually background patches very similar to their neighbors, have norms about
10 times larger and carry global information instead of the patch's own information. The model uses these redundant
patches as scratch space for its computation. Their fix: add a few **register** tokens, belonging to no patch, so
that the model has a place of its own for that. The idea is like GPT‑2's *attention sinks* in Lesson 17: softmax
forces each row to sum to 1, so the model needs a few tokens on which to put the attention it doesn't use. In this
small ViT‑Ti, those tokens don't pick background patches; they always sit in the four corners.

## 10 · Training a small ViT yourself

The two ViTs above trained for 8,000 steps on a many-core machine. To see how the learning works, we write
backpropagation for the very `forward` function of section 5 (the same steps as `reel/vit/vit.py`) and train a much
smaller ViT: 7×7 patches (16 tokens), $D = 32$, 2 blocks × 2 heads, AdamW, 400 steps × 64 images, less than half an
epoch over the 55,000 images. First, a gradient check with finite differences on a tiny model (float64).

In [ ]:
def layer_norm_back(dy, g, cache):
    hn, inv = cache
    dhn = dy * g
    dx = inv * (dhn - dhn.mean(-1, keepdims=True) - hn * (dhn * hn).mean(-1, keepdims=True))
    return dx, (dy * hn).reshape(-1, hn.shape[-1]).sum(0), dy.reshape(-1, hn.shape[-1]).sum(0)


def backward(p, cache, dz, heads):
    """The gradient of every parameter, going back through exactly the steps of forward(..., cache=True)."""
    xp, saved, hf, cf = cache
    B, N, _ = xp.shape
    T, Dm = N + 1, p["We"].shape[1]
    dh = Dm // heads
    f = lambda a: a.reshape(-1, a.shape[-1])  # (B, T, ·) → (B·T, ·)
    g = {"Wh": hf.T @ dz, "bh": dz.sum(0)}
    dcls, g["gf"], g["bf"] = layer_norm_back(dz @ p["Wh"].T, p["gf"], cf)
    dH = np.zeros((B, T, Dm), dz.dtype)
    dH[:, 0] = dcls  # only the [CLS] token goes into the classifier
    for l in reversed(range(len(saved))):
        a, c1, q, k, v, A, o, m, c2, u, gu, t = saved[l]
        w = {kk.split(".", 1)[1]: vv for kk, vv in p.items() if kk.startswith(f"{l}.")}
        g[f"{l}.W2"], g[f"{l}.c2"] = f(gu).T @ f(dH), f(dH).sum(0)  # MLP
        du = (dH @ w["W2"].T) * (0.5 * (1 + t) + 0.5 * u * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * u * u))
        g[f"{l}.W1"], g[f"{l}.c1"] = f(m).T @ f(du), f(du).sum(0)
        dx, g[f"{l}.g2"], g[f"{l}.b2"] = layer_norm_back(du @ w["W1"].T, w["g2"], c2)
        dH = dH + dx
        g[f"{l}.Wo"], g[f"{l}.bo"] = f(o).T @ f(dH), f(dH).sum(0)  # attention
        do = (dH @ w["Wo"].T).reshape(B, T, heads, dh).transpose(0, 2, 1, 3)
        dA = do @ v.transpose(0, 1, 3, 2)
        ds = A * (dA - (dA * A).sum(-1, keepdims=True)) * (1 / dh**0.5)  # through the softmax
        dqkv = np.stack([ds @ k, ds.transpose(0, 1, 3, 2) @ q, A.transpose(0, 1, 3, 2) @ do])
        dqkv = dqkv.transpose(1, 3, 0, 2, 4).reshape(B, T, 3 * Dm)
        g[f"{l}.Wqkv"], g[f"{l}.bqkv"] = f(a).T @ f(dqkv), f(dqkv).sum(0)
        dx, g[f"{l}.g1"], g[f"{l}.b1"] = layer_norm_back(dqkv @ w["Wqkv"].T, w["g1"], c1)
        dH = dH + dx
    if "pos" in p:
        g["pos"] = dH.sum(0)
    g["cls"] = dH[:, 0].sum(0)
    g["We"], g["be"] = xp.reshape(-1, xp.shape[-1]).T @ f(dH[:, 1:]), f(dH[:, 1:]).sum(0)
    return g


def softmax_ce(z, y):
    e = np.exp(z - z.max(1, keepdims=True))
    pr = e / e.sum(1, keepdims=True)
    loss = -np.log(pr[np.arange(len(y)), y]).mean()
    pr[np.arange(len(y)), y] -= 1
    return loss, pr / len(y)


def init_vit(P, Dm, L, M, seed, pos=True, dtype=np.float32):
    """As in reel/vit/vit.py: matrices ~ N(0, 1/number of inputs), positions ~ N(0, 0.02²), [CLS] and the classifier
    at 0."""
    rng = np.random.default_rng(seed)
    lin = lambda i, o: rng.normal(0, (1 / i) ** 0.5, (i, o)).astype(dtype)
    zeros, ones = (lambda n: np.zeros(n, dtype)), (lambda n: np.ones(n, dtype))
    p = {"We": lin(P * P, Dm), "be": zeros(Dm), "cls": zeros(Dm)}
    if pos:
        p["pos"] = rng.normal(0, 0.02, ((28 // P) ** 2 + 1, Dm)).astype(dtype)
    for l in range(L):
        p |= {f"{l}.g1": ones(Dm), f"{l}.b1": zeros(Dm), f"{l}.Wqkv": lin(Dm, 3 * Dm), f"{l}.bqkv": zeros(3 * Dm)}
        p |= {f"{l}.Wo": lin(Dm, Dm), f"{l}.bo": zeros(Dm), f"{l}.g2": ones(Dm), f"{l}.b2": zeros(Dm)}
        p |= {f"{l}.W1": lin(Dm, M), f"{l}.c1": zeros(M), f"{l}.W2": lin(M, Dm), f"{l}.c2": zeros(Dm)}
    return p | {"gf": ones(Dm), "bf": zeros(Dm), "Wh": np.zeros((Dm, 10), dtype), "bh": zeros(10)}


rng = np.random.default_rng(0)
tiny = init_vit(7, 8, 2, 12, seed=1, dtype=np.float64)
for key in tiny:  # move away from the symmetric starting point (classifier at 0, LayerNorm scales at 1)
    tiny[key] = tiny[key] + 0.3 * rng.standard_normal(tiny[key].shape)
Xg, yg = rng.random((3, 28, 28)), rng.integers(0, 10, 3)
z, cache = forward(tiny, Xg, P=7, heads=2, cache=True)
grads = backward(tiny, cache, softmax_ce(z, yg)[1], heads=2)
worst = 0.0
for key in tiny:
    for _ in range(3):
        idx = tuple(rng.integers(0, s) for s in tiny[key].shape)
        old = tiny[key][idx]
        tiny[key][idx] = old + 1e-5
        lp = softmax_ce(forward(tiny, Xg, P=7, heads=2), yg)[0]
        tiny[key][idx] = old - 1e-5
        lm = softmax_ce(forward(tiny, Xg, P=7, heads=2), yg)[0]
        tiny[key][idx] = old
        numeric = (lp - lm) / 2e-5
        worst = max(worst, abs(numeric - grads[key][idx]) / max(1e-5, abs(numeric) + abs(grads[key][idx])))
print(f"{len(tiny)} parameter arrays, 3 elements each: largest relative error {sci(worst)}")
assert worst < 1e-5

The gradients match the finite differences. The training loop below uses AdamW (weight decay 0.05 on the matrices),
warms the learning rate up over the first 5% of the steps, then decays it along a cosine, as the demo's script does.
We train two versions, with and without position embeddings, with the same seed and so the same batches of images,
then measure on the 10,000 test images.

In [ ]:
def train_vit(P=7, Dm=32, L=2, heads=2, M=64, steps=400, eta=4e-3, seed=0, pos=True, wd=0.05):
    p = init_vit(P, Dm, L, M, seed, pos)
    mom = {k: np.zeros_like(v) for k, v in p.items()}
    sq = {k: np.zeros_like(v) for k, v in p.items()}
    rng = np.random.default_rng(seed)
    warm = steps // 20
    for t in range(1, steps + 1):
        lr = eta * (t / warm if t <= warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / (steps - warm))))
        idx = rng.integers(0, 55000, 64)  # a batch of 64 images from train[:55k]
        z, cache = forward(p, Xtr[idx], P=P, heads=heads, cache=True)
        g = backward(p, cache, softmax_ce(z, ytr[idx])[1].astype(np.float32), heads)
        for k in p:  # AdamW
            mom[k] = 0.9 * mom[k] + 0.1 * g[k]
            sq[k] = 0.999 * sq[k] + 0.001 * g[k] ** 2
            if k.split(".")[-1][0] == "W":  # weight decay only on the matrices (We, Wqkv, Wo, W1, W2, Wh)
                p[k] *= 1 - lr * wd
            p[k] -= lr * (mom[k] / (1 - 0.9**t)) / (np.sqrt(sq[k] / (1 - 0.999**t)) + 1e-8)
    return p


Xtr = (train_img / 255.0).astype(np.float32)
ytr = train_lab.astype(np.int64)
small = {pos: train_vit(pos=pos) for pos in (True, False)}
acc_small = {pos: (predict(p, Xte, P=7, heads=2) == yte).mean() for pos, p in small.items()}
n_small = sum(a.size for a in small[True].values())
print(f"small ViT (7×7 patches, D = 32, 2 blocks, {num(n_small, 0)} parameters), 400 steps:")
print(f"  with positions {pct(acc_small[True])}, without positions {pct(acc_small[False])} on the 10,000 test images")
# 87.32% / 71.29% here; seeds 1–2 give 89.33% / 76.56% and 88.47% / 75.99%. float32 rounds differently on x86, hence
# the wide tolerance
assert abs(acc_small[True] - 0.8732) < 0.03 and acc_small[True] - acc_small[False] > 0.05

In only 400 steps, less than half an epoch, the small ViT already reads nearly nine tenths of the digits correctly;
without position embeddings it loses more than ten points.
The model of section 7 trained for 8,000 steps with $D = 48$ and 4 blocks to reach 97.83%. The question that
remains: with the same training recipe, how does ViT compare with a CNN, and how does that depend on the amount of
data?

## 11 · ViT and CNN: how much data is needed?

The demo's script trains ViT (4×4 and 7×7 patches), Lesson 12's LeNet and the MLP with one and the same recipe:
AdamW, 8,000 steps, batch size 64, the learning rate $\eta$ chosen on 5,000 validation images (`train[55k:]`, never
on the test set). Each training-set size from 1,000 to 55,000 images, 3 seeds. Every size gets the same number of
steps, so a small set is seen more times.

### 🤔 Predict first
With 1,000 training images, by how many percentage points does LeNet beat the ViT with 4×4 patches? At 55,000
images, does ViT overtake LeNet?

In [ ]:
SIZES = runs["setup"]["sizes"]
MODELS = [("vit4", "4×4-patch ViT", BLUE, "o"), ("vit7", "7×7-patch ViT", GREEN, "^"),
          ("lenet", "LeNet (CNN)", ORANGE, "s"), ("mlp", "MLP", GREY, "D")]  # fmt: skip


def tests(gid, n=55000):
    return [s["test"] for s in group(gid, n)["seeds"]]


med = {gid: np.array([np.median(tests(gid, n)) for n in SIZES]) for gid, *_ in MODELS}
print("images   " + "".join(f"{name:>15}" for _, name, *_ in MODELS) + "   LeNet − ViT 4×4")
for i, n in enumerate(SIZES):
    row = "".join(f"{pct(med[gid][i]):>15}" for gid, *_ in MODELS)
    print(f"{num(n, 0):>7}  {row}   {num(100 * (med['lenet'][i] - med['vit4'][i]), 1):>6} points")
bad = [s for s in group("mlp", 1000)["seeds"] if s["test"] < 0.5]
print(
    f"(MLP, 1,000 images: seed {bad[0]['seed']} failed to learn, {pct(bad[0]['test'])}; the table uses the median of 3 seeds)"
)
tr = {gid: min(s["train"] for s in group(gid, 1000)["seeds"]) for gid in ("vit4", "lenet")}
print(f"correct on the 1,000 training images themselves (worst seed): ViT {pct(tr['vit4'])}, LeNet {pct(tr['lenet'])}")

LeNet beats ViT at **every** dataset size. The gap narrows from 8.5 points at 1,000 images to 1.4 points at 55,000
images, but never reverses. At 1,000 images, both get every training image right: both memorize the data, and they
differ only in how they guess new images. The figure below plots the same table: each small marker is one seed, the
line connects the medians, and each model has its own color and marker shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for gid, name, col, mk in MODELS:
    for i, n in enumerate(SIZES):
        v = np.array(tests(gid, n))
        v = v[v > 0.5]  # the MLP's failed seed lies off the axis; see the printed line above
        ax.scatter([n] * len(v), 100 * v, color=col, marker=mk, s=14, alpha=0.5, lw=0)
    ax.plot(SIZES, 100 * med[gid], color=col, marker=mk, ms=7, lw=1.8, label=name)
ax.set(
    xscale="log", xlabel="training images (log scale)", ylabel="correct on the 10,000 test images (%)", ylim=(84, 100)
)
ax.set_title("The same training recipe, 3 seeds per point", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, loc="lower right")
fmt_axes(fig)
ax.set_xticks(SIZES, labels=[num(n, 0) for n in SIZES])
plt.show()

Convolution builds in two assumptions that ViT has to learn by itself: nearby pixels are related (locality), and a
stroke is the same stroke wherever it is (the filter is shared as it slides). That is an **inductive bias**. When
data is scarce, correct assumptions help generalization; when data is plentiful, the model learns most of them by
itself, so the gap shrinks.

The same file has a few side experiments, each with its own tuned $\eta$. We print the range over the 3 seeds and
compare it with the ViT with 4×4 patches at the same number of images; if the two ranges overlap, the difference is
within the seed-to-seed variation.

In [ ]:
def compare(gid, ref, n, n_ref=None, label=""):
    a, b = tests(gid, n), tests(ref, n_ref or n)
    if min(a) <= max(b) and min(b) <= max(a):
        verdict = "within the seed-to-seed variation"
    else:
        verdict = f"{'better by' if np.median(a) > np.median(b) else 'worse by'} {num(100 * abs(np.median(a) - np.median(b)), 1)} points"
    print(f"{label:<44} {span(a):>15}  vs. {span(b):>15}: {verdict}")


compare("vit7", "vit4", 55000, label="7×7 patches (55,000 images)")
compare("vit14", "vit4", 55000, label="14×14 patches, just 4 (55,000 images)")
compare("vit7", "vit4", 1000, label="7×7 patches (1,000 images)")
compare("vit4-gap", "vit4", 55000, label="mean pooling (GAP) instead of [CLS]")
compare("vit4-nopos", "vit4", 55000, label="no position embeddings")
compare("vit4-aug", "vit4", 5000, label="5,000 images + random shifts of ±2 px")
compare("vit4-aug", "vit4", 5000, 55000, label="5,000 images, ±2 px shifts vs. ViT on 55,000")
compare("lenet-aug", "lenet", 5000, label="LeNet, 5,000 images + ±2 px shifts")

Four things worth remembering:
- **Smaller patches are not automatically better.** On 28×28 digits, 7×7 and 14×14 patches both beat 4×4 at 55,000
  images, while being 3 and 11 times cheaper (section 6). At 1,000 images, 7×7 and 4×4 are level.
- **[CLS] is not required.** Averaging over all the patches (GAP) gives results on par with [CLS], as the ViT paper
  found (Appendix D.3), provided the learning rate is re-tuned.
- **Position matters:** removing the position embeddings costs nearly 9 points.
- **Data augmentation can stand in for part of the real data.** Just by shifting each image randomly by up to 2 px, a
  ViT trained on 5,000 images already matches a ViT trained on 55,000 images. LeNet improves too, but less, because
  convolution already has most of the robustness to shifts built in.

On real images, the ViT paper finds the same trend at a much larger scale. Trained only on ImageNet (1.3 million
images), ViT‑B/16 reaches 77.9% (Table 5) and trails ResNets (BiT) of the same size. Pretrained on ImageNet‑21k
(14 million images), the two are roughly level, and only with JFT‑300M (303 million images) does ViT pull ahead. But
“ViT always beats CNNs” is a misconception:
- DeiT (Touvron et al., 2020) got a ViT trained on ImageNet alone to 81.8% with a good recipe: strong data
  augmentation (RandAugment, Mixup, CutMix, random erasing), AdamW, 300 epochs. Adding knowledge distillation from a
  CNN teacher (RegNetY‑16GF, 82.9%) through a distillation token raises it to 83.4% (Tables 2–3 of DeiT).
- With the same amount of compute, modern convolutional networks keep up with ViT: ConvNeXt (Liu et al., 2022) on
  ImageNet, and NFNets pretrained on JFT‑4B (Smith et al., 2023) match ViTs with the same compute budget.

What ViT really brings is one architecture shared by images and text, easy to scale up, not a built-in higher
accuracy. Today ViT, often trained CLIP-style (Lesson 22), is the image encoder of most multimodal models that
publish their architecture.

## 12 · Shifting the image: ViT is closer to an MLP than to a CNN

The second inductive bias of convolution concerns shifts: shift the strokes and the feature map shifts along with
them, and max pooling can ignore small shifts (Lessons 11–12). ViT keeps that only for shifts by a multiple of the
patch side, thanks to using the same $E$ for every patch: a patch moved to another slot still gives the same token.
But each patch is tied to a fixed position embedding, so shifting the whole image by a few pixels makes the model see
completely different patches at completely different positions. The demo's script measured this on the 10,000 test
images, each shifted by 4 px in all four directions (no training image was shifted):

In [ ]:
for gid, name in (("lenet", "LeNet"), ("vit4", "ViT, positions"), ("mlp", "MLP"), ("vit4-nopos", "ViT, no positions")):
    g_ = group(gid)
    shift4 = [s["shift"]["acc"][4] for s in g_["seeds"]]
    print(f"{name:<17} original {span([s['test'] for s in g_['seeds']]):>15}  →  shifted 4 px {span(shift4):>15}")

Shifted by 4 px, LeNet keeps about two thirds, the ViT with positions less than half, close to the MLP: each patch is
tied to a fixed position, just as each pixel is tied to its own weight in the MLP. The ViT without positions loses
almost nothing. The cell below tries again on 300 test images, shifting them 2 px and 4 px to the right, and counts
the predictions that stay the same as on the original image.

In [ ]:
def shift(X, dx, dy):
    """Move the strokes dx px to the right and dy px down, filling the empty part with background (as in
    Lessons 11–12)."""
    out = np.zeros_like(X)
    H, W = X.shape[1:3]
    ys, yd = (slice(0, H - dy), slice(dy, H)) if dy >= 0 else (slice(-dy, H), slice(0, H + dy))
    xs, xd = (slice(0, W - dx), slice(dx, W)) if dx >= 0 else (slice(-dx, W), slice(0, W + dx))
    out[:, yd, xd] = X[:, ys, xs]
    return out


N_SH = 300  # the first test images
for name, p in vit32.items():
    for k in (2, 4):
        pr = predict(p, shift(Xte[:N_SH], k, 0))
        print(
            f"{name:>5}, shifted {k} px: {pct((pr == yte[:N_SH]).mean(), 1)} correct, "
            f"{pct((pr == pred[name][:N_SH]).mean(), 1)} of predictions the same as on the original"
        )
blank = Xte[:N_SH, :, 24:].max((1, 2)) == 0  # the 4 rightmost pixel columns (the last column of patches) are blank
Xb = Xte[:N_SH][blank]
gap4 = np.abs(forward(vit32["nopos"], shift(Xb, 4, 0)) - forward(vit32["nopos"], Xb)).max()
print(
    f"{int(blank.sum())}/{N_SH} images with an empty last column of patches: shifted 4 px, the logits of the ViT without positions change by at most {sci(gap4)}"
)
assert gap4 < 1e-4

For the ViT without positions, a 4 px shift keeps more predictions unchanged than a 2 px shift. The reason: 4 px is
exactly one patch. The image shifted by 4 px has exactly the old patches, only moved: the right-hand column of
patches falls off the edge (it is usually empty) and an empty column of patches comes in from the left. The bag of
patches doesn't change, so the logits don't change (the cell above: deviations at the level of float32 rounding
error). With a 2 px shift, the patch boundaries fall in the middle of the strokes, and every patch with ink becomes a
new patch. ViT is robust only to shifts that are exact multiples of the patch side; convolution slides pixel by
pixel, so it has no such limit.

## 13 · Try changing things

The cells below rerun section 10's `train_vit()` with other parameters (a few seconds each time). Change the values
and run them again.

### Large patches: are positions still needed?
With 14×14 patches, the image has only 4 patches, each a quadrant of the digit.

In [ ]:
STEPS_BIG = 300  # large patches learn fast; 300 steps are enough
big = {pos: train_vit(P=14, steps=STEPS_BIG, pos=pos) for pos in (True, False)}
acc_big = {pos: (predict(p, Xte, P=14, heads=2) == yte).mean() for pos, p in big.items()}
print(f"14×14 patches, {STEPS_BIG} steps: with positions {pct(acc_big[True])}, without positions {pct(acc_big[False])}")
print(
    f"(7×7 patches, 400 steps in section 10: with positions {pct(acc_small[True])}, without positions {pct(acc_small[False])})"
)
assert abs(acc_big[True] - acc_big[False]) < 0.02

With 4 large patches, removing the positions costs almost nothing. MNIST digits are always centered, so the content
of a quadrant alone (which way a curved stroke opens, which edge it hugs) is enough to tell which quadrant it is. The
larger the patches, the more each patch carries information about its own position; the smaller the patches, the
more the model needs $E_{\text{pos}}$.

Other things worth trying:
- `train_vit(seed=1)`, `seed=2`: each seed gives a slightly different result. Compare two configurations only when
  the difference is larger than that spread.
- `train_vit(steps=1000)`: takes 2.5 times as long; the accuracy keeps rising.
- `train_vit(P=4)`: 49 small patches, each step much more expensive (section 6), and in 400 steps it learns more
  slowly than with 7×7 patches.
- In `forward`, replace the [CLS] token with the average of all patches (`hf = layer_norm(h[:, 1:].mean(1), ...)`)
  and change `backward` to match: that is the GAP of section 11.

## 14 · Limits and next steps

- **The cost grows with the square of the number of tokens.** High-resolution images or small patches give a great
  many tokens. Variants such as Swin reuse part of the idea of convolution (attention within local windows) to
  reduce that cost.
- **ViT needs data, or data augmentation.** Without the biases of convolution, it has to learn locality and
  robustness to shifts by itself; on MNIST, with the same recipe, LeNet wins at every dataset size we tried.
- **Attention maps are easy to misread.** The attention of [CLS] at the last layer piles up on corner tokens that
  have nothing to do with the object in the image; it tells which way the information goes, not why the model
  predicts what it does.

ViT's strength is its simplicity: the image becomes a sequence of tokens, then the very Transformer used for text.
That is how the same architecture can learn from hundreds of millions of image–caption pairs; Lesson 22 runs CLIP's
ViT‑B/32 image encoder, trained that way.

Up to here, every image model in the course reads an image to produce a label. The next chapter goes the other way:
from a few numbers, it generates an image. Lesson 19 starts with the autoencoder, a network that compresses a
784-pixel digit into a few numbers and then rebuilds it.

## Summary
- ViT cuts the image into $N = HW/P^2$ patches: a 28×28 digit with 4×4 patches gives 49 patches + [CLS] = 50
  tokens; a 224×224 image with 16×16 patches gives 197 tokens.
- Flattening the patch and multiplying by $E$ is exactly a convolutional layer with $D$ filters of size
  $P \times P$ and stride $P$ (differences around $10^{-16}$, only rounding error).
- The demo's two ViTs (79,690 parameters): with positions 97.83% correct, without positions 89.07%. Without
  positions, shuffling the patches doesn't change a single logit: the model sees only a bag of patches.
- The MNIST ViT relies heavily on position (with shuffled patches it keeps 18.76–21.01%) without learning the 2‑D
  grid; ViT‑Ti/16 on ImageNet does learn it (196/196 patches are most similar to an adjacent patch).
- ViT‑Ti has near-looking (26 px) and far-looking heads already in block 1; from block 8 on, 3–4 tokens at the
  corners of the image have high norms, and in block 12 they receive 22–53% of [CLS]'s attention.
- With the same training recipe, LeNet beats ViT at every dataset size (8.5 points at 1,000 images, 1.4 points at
  55,000 images); ±2 px shifts during training help ViT more than they help LeNet.

## Review questions
1. How many tokens does a 384×384 image with 16×16 patches give? If the patches shrink to 8×8, roughly how many
   times larger does the attention matrix get?
2. Why is the patch projection $x_p E$ a convolutional layer? What are its filters, its stride and its padding?
3. Without position embeddings, how do the ViT's logits change when the order of the patches is shuffled? Why?
4. The MNIST model with positions collapses when the patches are shuffled, yet its position embeddings don't
   resemble a 2‑D grid. Do these two facts contradict each other?
5. Why does the ViT without positions keep its predictions better when the image is shifted by 4 px than by 2 px?
6. Under what conditions is “ViT is better than CNNs” true? What do the MNIST results and the DeiT paper say about
   it?

<details><summary>Hints and answers</summary>

1. $(384/16)^2 + 1 = 577$ tokens. 8×8 patches give $48^2 + 1 = 2{,}305$ tokens, and the $T \times T$ attention
   matrix gets about 16 times larger.
2. Column $d$ of $E$ holds $P^2 C$ weights; rearranged into $P \times P \times C$, it is a filter. Multiplying the
   patch by that column is the convolution at one position. The stride is $P$ and there is no padding, so the
   windows fit the patch grid exactly, without overlapping.
3. They don't change (apart from rounding error). Every per-token computation is the same for all tokens, and
   attention is a weighted sum over the set of tokens, so swapping the inputs only swaps the outputs; [CLS] is not a
   patch, so its output stays the same.
4. No. The model needs to know which patch is in which square (each vector is like a separate house number), but not
   which square is next to which, because MNIST digits are always centered. In real photos, objects can be
   anywhere, so ViT‑Ti learns the neighborhood relations.
5. 4 px is exactly the side of a patch: the shifted image has the same patches, only moved, so the bag of patches
   doesn't change. A 2 px shift cuts the strokes along different lines, giving new patches.
6. When there is a great deal of pretraining data (the ViT paper: JFT‑300M) or a strong training recipe (DeiT: data
   augmentation, AdamW, 300 epochs, distillation). On MNIST with the same recipe, LeNet wins at every dataset size;
   with the same amount of compute, modern CNNs (ConvNeXt, NFNet) keep up with ViT.
</details>

## Further reading
- A. Dosovitskiy et al. (2020), [An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929): the ViT paper
  (equation 1, inductive bias in section 3.1, attention distance in Figure 7, the position ablation in Table 8).
- Google Research's original code and weights: [vision_transformer](https://github.com/google-research/vision_transformer)
  (the patch projection written with `nn.Conv`).
- H. Touvron et al. (2020), [DeiT: Training data-efficient image transformers](https://arxiv.org/abs/2012.12877).
- A. Steiner et al. (2021), [How to train your ViT?](https://arxiv.org/abs/2106.10270): data augmentation,
  regularization and more compute can stand in for about ten times as much data; the source of the demo's ViT‑Ti/16
  ([timm](https://huggingface.co/timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k)).
- M. Raghu et al. (2021), [Do Vision Transformers See Like Convolutional Neural Networks?](https://arxiv.org/abs/2108.08810).
- J.-B. Cordonnier, A. Loukas & M. Jaggi (2019), [On the Relationship between Self-Attention and Convolutional Layers](https://arxiv.org/abs/1911.03584).
- S. Abnar & W. Zuidema (2020), [Quantifying Attention Flow in Transformers](https://arxiv.org/abs/2005.00928): attention rollout.
- T. Darcet et al. (2023), [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588).
- Z. Liu et al. (2022), [A ConvNet for the 2020s](https://arxiv.org/abs/2201.03545) (ConvNeXt), and S. Smith et al.
  (2023), [ConvNets Match Vision Transformers at Scale](https://arxiv.org/abs/2310.16764).
- d2l.ai, [Transformers for Vision](https://d2l.ai/chapter_attention-mechanisms-and-transformers/vision-transformer.html):
  a ViT written in PyTorch, trained on Fashion‑MNIST.

**Next lesson:** [Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/):
compressing an image into a few numbers, then learning to generate new images from them.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")